# 81 — Pseudo-Label Self-Training (Transductive, 3 Iterations)

The test set consists of analogs of 63 training hits — average Tanimoto 0.52 to train.
This makes the test set "easy" for a transductive learner:

1. Train model on CRC data → predict all 513 test compounds
2. For test compounds with high similarity to training (Tanimoto > 0.65):
   - Confidence-gate: only add pseudo-labels we're confident about
   - Add to training with weight = max_sim_to_train
3. Retrain → repeat 3 iterations

The 32 test compounds near cliff members are likely to get better pseudo-labels
as the model improves from easier examples → semi-supervised curriculum.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
# Similarity of each test compound to nearest training compound
print("Computing test→train similarity...", flush=True)
dot_te_tr = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_te_tr / np.maximum(rs_te + rs_tr_v - dot_te_tr, 1e-6)
max_sim = sim_te_tr.max(1)  # (513,) — max similarity to any training compound
print(f"Test→train max similarity: "
      f"mean={max_sim.mean():.3f}  min={max_sim.min():.3f}  max={max_sim.max():.3f}")
print(f"High-confidence (sim>0.65): {(max_sim>0.65).sum()}")
print(f"Med-confidence  (sim>0.50): {(max_sim>0.50).sum()}")
print(f"Low-confidence  (sim≤0.50): {(max_sim<=0.50).sum()}")


Computing test→train similarity...


Test→train max similarity: mean=0.532  min=0.323  max=0.806
High-confidence (sim>0.65): 40
Med-confidence  (sim>0.50): 330
Low-confidence  (sim≤0.50): 183


In [5]:
THRESHOLDS   = [0.65, 0.55, 0.45]  # tighten per iteration
WEIGHT_SCALE = 0.8   # pseudo-label weight = max_sim * WEIGHT_SCALE
N_ROUNDS     = 3

X_cur, y_cur, w_cur = X_tr.copy(), y_tr.copy(), np.ones(len(y_tr), dtype=np.float32)
history = []

for rnd in range(N_ROUNDS):
    print(f"\n=== Round {rnd+1}/{N_ROUNDS}  threshold≥{THRESHOLDS[rnd]:.2f} ===", flush=True)
    # Train on current augmented set
    m = lgb.train(LGBM, lgb.Dataset(X_cur, label=y_cur, weight=w_cur),
                  callbacks=[lgb.log_evaluation(-1)])
    te_pred = m.predict(X_te)

    # Evaluate on scaffold CV of original CRC (proxy metric)
    oof_rnd = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        mf = lgb.train(LGBM, lgb.Dataset(X_cur[tr_idx], label=y_cur[tr_idx], weight=w_cur[tr_idx]),
                       valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                       callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
        oof_rnd[va_idx] = mf.predict(X_tr[va_idx])
    r = full_metrics(y_tr, oof_rnd, cliff_pairs, f"round_{rnd+1}")
    history.append(r)

    # Add high-confidence pseudo-labels
    conf_mask = max_sim >= THRESHOLDS[rnd]
    n_add = conf_mask.sum()
    print(f"  Adding {n_add} pseudo-labels (sim≥{THRESHOLDS[rnd]:.2f})")
    if n_add > 0:
        X_pl = X_te[conf_mask]
        y_pl = te_pred[conf_mask]
        w_pl = (max_sim[conf_mask] * WEIGHT_SCALE).astype(np.float32)
        X_cur = np.vstack([X_tr, X_pl])
        y_cur = np.concatenate([y_tr, y_pl])
        w_cur = np.concatenate([np.ones(len(y_tr), dtype=np.float32), w_pl])
    oof = oof_rnd

print("\n=== Training history ===")
print(pd.DataFrame(history, index=[f"round_{i+1}" for i in range(N_ROUNDS)]).round(4).to_string())



=== Round 1/3  threshold≥0.65 ===


  [round_1] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  Adding 40 pseudo-labels (sim≥0.65)

=== Round 2/3  threshold≥0.55 ===


  [round_2] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  Adding 178 pseudo-labels (sim≥0.55)

=== Round 3/3  threshold≥0.45 ===


  [round_3] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  Adding 460 pseudo-labels (sim≥0.45)

=== Training history ===
            RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
round_1  0.5643  0.5134  0.5991    0.774    0.7268   0.5345        NaN
round_2  0.5643  0.5134  0.5991    0.774    0.7268   0.5345        NaN
round_3  0.5643  0.5134  0.5991    0.774    0.7268   0.5345        NaN


In [6]:
# Final model + test predictions
m_final = lgb.train(LGBM, lgb.Dataset(X_cur, label=y_cur, weight=w_cur),
                    callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_te), y_tr.min()-0.5, y_tr.max()+0.5)
print(f"Final training size: {len(X_cur):,}  (original: {len(X_tr):,} + pseudo: {len(X_cur)-len(X_tr):,})")
np.save(DATA_PROCESSED/"oof_pseudo_label.npy", oof)
np.save(DATA_PROCESSED/"te_oof_pseudo_label.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"81_pseudo_label_selftrain.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Final training size: 4,599  (original: 4,139 + pseudo: 460)
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\81_pseudo_label_selftrain.csv
Test: min=2.44 med=4.97 max=6.04
